# NB6 - Feature Ablation and SHAP Diagnostics

Reports optional formula-AS feature ablations and SHAP diagnostics. Ablations run in parallel by feature set; SHAP is optional post-processing after B1 training.

## Snellius Pipeline Mode

This notebook is now a reporting/orchestration notebook. Heavy training and per-day evaluation are executed by Slurm scripts under `scripts/snellius/`; this notebook reads the resulting artifacts.

Expected artifacts:
- `results/formula_feature_ablation_results.csv`
- `results/b1_shap_feature_importance.csv`

Run from MobaXterm/Snellius with:

```bash
cd /home/<user>/thesis
export PROJECT_DIR=/home/<user>/thesis
export DATA_DIR=/scratch-shared/<user>/datasets
export CONDA_ENV=mysimenv
bash scripts/snellius/submit_final_pipeline.sh
```

In [ ]:
import json
import pathlib
import sys

import pandas as pd
import matplotlib.pyplot as plt

PROJECT_ROOT = next(
    (p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "procs").exists()),
    pathlib.Path.cwd(),
)
RESULTS_DIR = PROJECT_ROOT / "results"
MODELS_DIR = PROJECT_ROOT / "models"
META_DIR = RESULTS_DIR / "job_metadata"

pd.set_option("display.float_format", "{:.6f}".format)


def require_file(path, label=None, strict=False):
    path = pathlib.Path(path)
    if path.exists():
        print(f"OK      {label or path.name}: {path}")
        return True
    message = f"MISSING {label or path.name}: {path}"
    if strict:
        raise FileNotFoundError(message)
    print(message)
    return False


def read_csv(path, **kwargs):
    path = pathlib.Path(path)
    require_file(path, strict=True)
    return pd.read_csv(path, **kwargs)


def metric_summary(df):
    metrics = [c for c in ["Sharpe", "Sortino", "Max DD", "P&L-to-MAP", "Final PnL", "Mean |q|", "Near Cap Fraction"] if c in df.columns]
    return pd.DataFrame({"Mean": df[metrics].mean(), "Median": df[metrics].median(), "Std": df[metrics].std(ddof=1)})


def formula_metadata_files():
    if not META_DIR.exists():
        return []
    return sorted(META_DIR.glob("*.json"))

print(f"Project root : {PROJECT_ROOT}")
print(f"Results dir  : {RESULTS_DIR}")
print(f"Models dir   : {MODELS_DIR}")
print("Official Snellius execution path: bash scripts/snellius/submit_final_pipeline.sh")

## Artifact Preflight

In [ ]:
require_file(RESULTS_DIR / 'formula_feature_ablation_results.csv')
require_file(RESULTS_DIR / 'b1_shap_feature_importance.csv')

## Ablation Results

In [ ]:
ablation_path = RESULTS_DIR / "formula_feature_ablation_results.csv"
if ablation_path.exists():
    df_ablation = read_csv(ablation_path)
    display(df_ablation)
else:
    print("No ablation summary yet. Run RUN_NB6=1 bash scripts/snellius/submit_final_pipeline.sh")

## SHAP Diagnostics

In [ ]:
shap_path = RESULTS_DIR / "b1_shap_feature_importance.csv"
if shap_path.exists():
    df_shap = read_csv(shap_path)
    display(df_shap.sort_values("mean_abs_shap", ascending=False).head(20))
    pivot = df_shap.pivot(index="feature", columns="output", values="mean_abs_shap")
    pivot.plot(kind="bar", figsize=(10, 5), title="B1 policy SHAP feature importance")
    plt.tight_layout()
    plt.savefig(RESULTS_DIR / "nb6_shap_importance_report.png", dpi=150, bbox_inches="tight")
    plt.show()
else:
    print("No SHAP output yet. Set RUN_SHAP=1 for optional SHAP post-processing.")